# Data Ethics — Bias, Fairness, and Responsible Data Science

## Introduction

Data science tools are powerful — and that power creates responsibility. Biased data leads to biased models. Biased models can harm real people. This notebook surveys the key concepts in data ethics: where bias comes from, how to measure fairness, and how to practice responsible data science.

## Objectives

You will be able to:

* Identify the main types of bias in data and models
* Understand the legal and ethical framework around protected attributes
* Recognize tradeoffs between different fairness definitions
* Apply practical steps for responsible EDA and modeling
* Cite real-world examples where data ethics failures caused harm

---

## Where Does Bias Come From?

### 1. Historical Bias

The world has never been perfectly fair. If you train a model on historical hiring decisions, and those decisions were biased against women or minorities, the model will learn — and perpetuate — that bias.

**Example:** Amazon's internal resume-screening tool (2018) downgraded résumés that contained the word "women's" (e.g., "women's chess club") because it was trained on 10 years of historically male-dominated hiring data.

### 2. Sampling Bias

When the data collection process systematically excludes certain groups, the resulting dataset doesn't represent the population.

**Example:** Medical imaging datasets are predominantly sourced from academic hospitals in wealthy countries, meaning skin cancer detection models perform significantly worse on darker skin tones.

### 3. Measurement Bias

When the way we measure a concept introduces distortion. Often occurs when a proxy variable doesn't mean the same thing across groups.

**Example:** Using ZIP code as a proxy for creditworthiness encodes racial residential segregation into lending decisions.

### 4. Aggregation Bias

Building a single model for a diverse population when different subgroups have different underlying patterns.

**Example:** A diabetes prediction model trained on a general population may underperform for specific ethnic groups with different disease progression patterns.

### 5. Feedback Loop Bias

Model predictions change the real world, which then becomes the next round of training data.

**Example:** Predictive policing sends more officers to neighborhoods already over-policed, leading to more arrests there, which confirms the model's predictions — regardless of actual crime rates.

---

## Protected Attributes

In [ ]:
import pandas as pd

# In most jurisdictions, using these attributes in automated decisions is
# either illegal or heavily regulated:

protected_attributes = {
    'Race / Ethnicity': 'Fair Housing Act, Equal Credit Opportunity Act',
    'Sex / Gender':     'Title VII, Equal Pay Act',
    'Age':              'Age Discrimination in Employment Act (40+)',
    'Religion':         'Title VII',
    'Disability':       'Americans with Disabilities Act',
    'National origin':  'Title VII',
    'Pregnancy':        'Pregnancy Discrimination Act',
    'Marital status':   'Equal Credit Opportunity Act',
}

df_laws = pd.DataFrame(list(protected_attributes.items()),
                       columns=['Attribute', 'Key US Legislation'])
print(df_laws.to_string(index=False))

**Proxy discrimination:** Even if you remove the protected attribute from the model, correlated features can encode the same information. ZIP code correlates with race; shopping habits correlate with gender. Removing the variable is not enough — you must test outcomes.

---

## Measuring Fairness

In [ ]:
import numpy as np

# Simulated loan application outcomes
np.random.seed(42)
n = 500

df_loans = pd.DataFrame({
    'group':      np.random.choice(['Group A', 'Group B'], n, p=[0.6, 0.4]),
    'credit_score': np.random.randint(500, 800, n),
    'income':     np.random.normal(60000, 20000, n).clip(20000, 150000).round(-3).astype(int),
})

# Simulate biased approval: Group B gets harder threshold
df_loans['approved'] = (
    (df_loans['credit_score'] > 640) |
    ((df_loans['credit_score'] > 620) & (df_loans['group'] == 'Group A'))
).astype(int)

# Demographic parity — equal approval rates across groups
approval_rates = df_loans.groupby('group')['approved'].mean()
print("Approval rates by group:")
print(approval_rates.round(3))

disparity = approval_rates['Group A'] / approval_rates['Group B']
print(f"\nDisparate impact ratio: {disparity:.2f}")
print("(The '80% rule': ratio < 0.8 suggests adverse impact)")

### Competing Fairness Definitions

There is no single definition of fairness. Each captures a different value, and they are often **mathematically incompatible** with each other:

| Definition | Meaning | When to prioritize |
|------------|---------|--------------------|
| **Demographic parity** | Equal approval rates across groups | When base rates should not matter (e.g., housing) |
| **Equal opportunity** | Equal true positive rates | When cost of false negatives differs by group |
| **Equalized odds** | Equal TPR and FPR | High-stakes decisions (criminal justice) |
| **Calibration** | Predicted probability = true probability | When probability estimates are used directly |
| **Individual fairness** | Similar individuals get similar outcomes | When comparison is meaningful |

**Chouldechova (2017)** proved that demographic parity, equal opportunity, and calibration cannot all hold simultaneously when base rates differ between groups. You must choose.

---

## Privacy and Data Minimization

### Key Principles

1. **Data minimization** — collect only what you need. Don't store sensitive attributes unless they're necessary.

2. **Purpose limitation** — data collected for one purpose should not be used for another.

3. **Anonymization limits** — de-identified data can often be re-identified. Latanya Sweeney showed that 87% of Americans can be uniquely identified by ZIP code, birthdate, and sex alone.

4. **Differential privacy** — a mathematical framework that adds calibrated noise to query results so that individual records cannot be inferred.

5. **Right to explanation** — the EU's GDPR gives individuals the right to a meaningful explanation of automated decisions that affect them.

---

## Practical Checklist for Responsible Data Science

In [ ]:
checklist = [
    ('Before collecting data',
     ['Do we have consent? Is collection purpose clear?',
      'What sensitive attributes are present or inferable?',
      'Is the collection process representative?']),
    ('During EDA',
     ['Examine distributions by protected group',
      'Check for class imbalance within subgroups',
      'Identify proxy variables for protected attributes']),
    ('Before modeling',
     ['Define the fairness metric appropriate for this use case',
      'Establish a baseline and which group should not be disadvantaged',
      'Consider who bears the cost of errors (false positives vs false negatives)']),
    ('After modeling',
     ['Measure accuracy, precision, recall separately per protected group',
      'Compute disparate impact ratio',
      'Document model decisions, data sources, and limitations']),
    ('Deployment',
     ['Monitor model performance over time by subgroup',
      'Establish a process for individuals to contest decisions',
      'Plan for periodic audits and model retraining']),
]

for stage, items in checklist:
    print(f"\n{'─' * 50}")
    print(f" {stage}")
    print(f"{'─' * 50}")
    for item in items:
        print(f"  ☐ {item}")

---

## Case Studies

### COMPAS Recidivism Algorithm (2016)
ProPublica's analysis found that the COMPAS algorithm, used by courts to predict recidivism, was twice as likely to falsely flag Black defendants as high risk compared to white defendants. Northpointe (the developer) responded that the algorithm was *calibrated* — an example of the fundamental fairness tension above.

### Facial Recognition (2018–present)
MIT Media Lab researchers (Buolamwini & Gebru) found commercial facial recognition systems had error rates of <1% for light-skinned males but up to 35% for dark-skinned females. Several US cities have banned facial recognition use by police as a result.

### Healthcare Risk Algorithm (2019)
A widely used algorithm that assigned patients priority for care management programs showed systemic bias against Black patients — not because race was used as input, but because it used healthcare *cost* as a proxy for *need*. Since Black patients historically had less access to care, they had lower costs even when sicker.

---

## Further Reading

- **"Weapons of Math Destruction"** — Cathy O'Neil (accessible book-length treatment)
- **"The Alignment Problem"** — Brian Christian (AI alignment and values)
- **Fairness and Machine Learning** — Barocas, Hardt & Narayanan (free online textbook)
- **EU AI Act** — the world's first comprehensive AI regulation framework

## Summary

Bias enters data at collection, labeling, and aggregation. No single fairness metric captures all of what we mean by "fair" — choosing one involves value judgments that belong in the open. Responsible data science means auditing for disparate impact, documenting decisions, and building mechanisms for human oversight into automated systems.